In [1]:
import numpy as np
import gymnasium as gym
import imageio
from tqdm import tqdm

## 1. Functions

In [ ]:
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

def epsilon_greedy_policy(Qtable, state, epsilon):
    if np.random.uniform(0, 1) < epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Qtable[state, :])
    return action

def greedy_policy(Qtable, state):
    return np.argmax(Qtable[state, :])

def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, _ = env.reset()
        
        for step in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)
            new_state, reward, terminated, truncated, _ = env.step(action)
            Qtable[state, action] = Qtable[state, action] + learning_rate * (
                reward + gamma * np.max(Qtable[new_state, :]) - Qtable[state, action]
            )
            state = new_state
            
            if terminated or truncated:
                break
    return Qtable

def evaluate_agent(env, max_steps, n_eval_episodes, Qtable):
    episode_rewards = []
    
    for episode in range(n_eval_episodes):
        state, _ = env.reset()
        total_rewards = 0
        
        for step in range(max_steps):
            action = greedy_policy(Qtable, state)
            new_state, reward, terminated, truncated, _ = env.step(action)
            total_rewards += reward
            state = new_state
            
            if terminated or truncated:
                break
        
        episode_rewards.append(total_rewards)
    
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)
    
    return mean_reward, std_reward

def record_video(env, Qtable, video_path, fps=2):
    frames = []
    state, _ = env.reset()
    img = env.render()
    frames.append(img)
    
    done = False
    while not done:
        action = greedy_policy(Qtable, state)
        state, reward, terminated, truncated, _ = env.step(action)
        img = env.render()
        frames.append(img)
        done = terminated or truncated
    
    imageio.mimsave(video_path, frames, fps=fps)



## 2. Environment

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

state_space = env.observation_space.n
action_space = env.action_space.n

Qtable_taxi = initialize_q_table(state_space, action_space)


In [ ]:
n_training_episodes = 20000
learning_rate = 0.7
n_eval_episodes = 100
max_steps = 200
gamma = 0.95
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.0005


In [ ]:
Qtable_taxi = train(
    n_training_episodes,
    min_epsilon,
    max_epsilon,
    decay_rate,
    env,
    max_steps,
    Qtable_taxi
)


In [ ]:
mean_reward, std_reward = evaluate_agent(
    env,
    max_steps,
    n_eval_episodes,
    Qtable_taxi
)

print(f"Resultado de la evaluación: Media={mean_reward:.2f} +/- {std_reward:.2f}")

video_path = "taxi_replay.gif"
record_video(env, Qtable_taxi, video_path, fps=2)

env.close()